# Distribution of Returns Analysis

Run the price fetch step by step and visualize each transformation.
Set `ticker` and `timeframe` below.

In [ ]:
import sys
from pathlib import Path

import pandas as pd

sys.path.insert(0, str(Path.cwd().parent / 'scripts'))

from dor import fetch_prices, save_results, drop_today_row, KEEP_COLUMNS

## Fetch and Manage Data

### 1. Fetch raw data

In [ ]:
ticker = 'AAPL'
timeframe = 'd'  # d | w | m

df = fetch_prices(ticker, timeframe)
print(f'{len(df)} rows from {df.index.min().date()} to {df.index.max().date()}')
df.tail()

In [ ]:
ax = df['Close'].plot(title=f'{ticker} ({timeframe}) - Close', figsize=(12, 5))
ax.set_ylabel('Close')
ax.grid(True, alpha=0.3)

### 2. Reformat the date index as `DD-MM-YY`

In [ ]:
cleaned = df.copy()
cleaned.index = cleaned.index.strftime('%d-%m-%y')
cleaned.index.name = 'Date'
cleaned.head()

### 3. Keep only the columns we need

In [ ]:
cleaned = cleaned[KEEP_COLUMNS]
cleaned.head()

### 4. Drop today's row if present

Today's bar is still forming, so its OHLC values aren't final. Drop it so all returns are computed from settled sessions only.

In [ ]:
before = len(cleaned)
cleaned = drop_today_row(cleaned)
print(f'Dropped {before - len(cleaned)} row(s). New size: {len(cleaned)} rows.')
cleaned.head()

### 5. Sort from newest to oldest

The index is now a `DD-MM-YY` string, so a plain lexicographic sort would be wrong.
Parse the strings back to dates inside the sort to keep chronological order.

In [ ]:
cleaned = cleaned.sort_index(
    ascending=False,
    key=lambda idx: pd.to_datetime(idx, format='%d-%m-%y'),
)
cleaned.head()

### 6. C-C Returns (current Adj Close vs previous period's Adj Close)

Frame is newest-first, so the *previous* period sits in the row below — use `shift(-1)`.

In [ ]:
prev_adj_close = cleaned['Adj Close'].shift(-1)
cleaned['C-C Returns'] = (cleaned['Adj Close'] - prev_adj_close) / prev_adj_close * 100
cleaned[['Adj Close', 'C-C Returns']].head()

### 7. H-L Returns (low to high of the period)

In [ ]:
cleaned['H-L Returns'] = (cleaned['High'] - cleaned['Low']) / cleaned['Low'] * 100
cleaned[['High', 'Low', 'H-L Returns']].head()

### 8. O-C Returns (open to close) — daily only

Open-to-close only makes sense for daily bars. For weekly/monthly the "open" and "close" span many sessions, so the column is skipped.

In [ ]:
if timeframe == 'd':
    cleaned['O-C Returns'] = (cleaned['Close'] - cleaned['Open']) / cleaned['Open'] * 100
    display(cleaned[['Open', 'Close', 'O-C Returns']].head())
else:
    print(f"Skipped O-C Returns: only added for daily data (current timeframe: '{timeframe}').")

### 9. Drop NaN rows

Only the oldest row should have a `NaN` (in `C-C Returns`, since there's no prior period). Verify the count, then drop it.

In [ ]:
nan_per_col = cleaned.isna().sum()
rows_with_nan = cleaned.isna().any(axis=1).sum()
print(f'Rows with at least one NaN: {rows_with_nan}')
print('NaN per column:')
print(nan_per_col)

In [ ]:
before = len(cleaned)
cleaned = cleaned.dropna()
print(f'Dropped {before - len(cleaned)} row(s). New size: {len(cleaned)} rows.')
cleaned.tail()

### 10. Save raw + cleaned CSVs and summary JSON

In [ ]:
raw_csv_path, clean_csv_path, report_path, summary = save_results(ticker, timeframe, df)
summary